# Biggest options flow trades on a day

Pulls Unusual Whales flow alerts for one trading day, labels each trade bullish or bearish by who
initiated it, and ranks the biggest. Then checks what the stock did afterwards.

- **Bullish:** calls bought at the ask, or puts sold at the bid.
- **Bearish:** puts bought at the ask, or calls sold at the bid.
- **Neutral:** premium split near the mid, or mostly unclassified.

Opening trades (new positions), sweeps (urgent fills across exchanges), short expiries and
out-of-the-money strikes look most like real bets; big deep-in-the-money, floor or multi-leg trades
are often hedges or spreads. Market data only: nothing here reads an account or places an order.

Run it from the TradingAgents folder's `.venv` (`.venv/bin/jupyter lab`), so the keys in `.env` load.

In [ ]:
from datetime import datetime, timedelta

import pandas as pd

from tradingagents.dataflows.vendors import tradier, unusual_whales

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## Settings

In [ ]:
DAY = "2026-10-19"          # trading day to look at
MIN_PREMIUM = 500_000       # skip alerts under this premium, in dollars
TICKER = None               # e.g. "AAPL" for one name, None for the whole market
TOP = 25                    # rows to show
LEAN = 0.6                  # share of premium on one side needed to call it bullish or bearish

## Pull and label the flow

In [ ]:
def fetch_flow(day: str, min_premium: int = MIN_PREMIUM, ticker: str | None = None, max_pages: int = 25) -> pd.DataFrame:
    """Every flow alert on ``day`` at or above ``min_premium``, newest first, paging back through the day."""
    params = {"newer_than": f"{day}T00:00:00Z", "older_than": f"{day}T23:59:59Z", "limit": 200,
              "min_premium": int(min_premium)}
    if ticker:
        params["ticker_symbol"] = ticker.upper()
    rows = []
    for _ in range(max_pages):
        page = unusual_whales._request("option-trades/flow-alerts", params)
        rows += page
        if len(page) < params["limit"]:
            break
        params = {**params, "older_than": min(str(r.get("created_at", "")) for r in page)}
    else:
        print(f"Stopped after {max_pages} pages; raise MIN_PREMIUM or max_pages for the full day.")
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df.drop_duplicates(subset=[c for c in ("id",) if c in df] or None)
    return df[df["created_at"].astype(str).str[:10] == day].reset_index(drop=True)


def label(df: pd.DataFrame, lean: float = LEAN) -> pd.DataFrame:
    """Adds lean (bullish/bearish/neutral), signed premium, days to expiry and moneyness."""
    df = df.copy()
    for col in ("total_premium", "total_ask_side_prem", "total_bid_side_prem", "strike", "underlying_price",
                "volume_oi_ratio", "total_size"):
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    sided = df["total_ask_side_prem"].fillna(0) + df["total_bid_side_prem"].fillna(0)
    ask_share = (df["total_ask_side_prem"].fillna(0) / sided).where(sided > 0)
    bought, sold = ask_share >= lean, ask_share <= 1 - lean
    call = df["type"] == "call"
    df["lean"] = "neutral"
    df.loc[(call & bought) | (~call & sold), "lean"] = "bullish"
    df.loc[(call & sold) | (~call & bought), "lean"] = "bearish"
    df["ask_share"] = ask_share.round(2)
    df["signed_premium"] = df["total_premium"] * df["lean"].map({"bullish": 1, "bearish": -1, "neutral": 0})
    df["dte"] = (pd.to_datetime(df["expiry"]) - pd.to_datetime(df["created_at"].astype(str).str[:10])).dt.days
    # Out of the money is positive: calls above the price, puts below it.
    df["otm_pct"] = ((df["strike"] / df["underlying_price"] - 1) * df["type"].map({"call": 1, "put": -1}) * 100).round(1)
    for flag in ("all_opening_trades", "has_sweep", "has_floor", "has_multileg"):
        df[flag] = df[flag].fillna(False).astype(bool) if flag in df else False
    # A rough "looks like a bet" flag: opening, not a spread, out of the money, within 60 days.
    df["bet_like"] = df["all_opening_trades"] & ~df["has_multileg"] & (df["otm_pct"] > 0) & (df["dte"] <= 60)
    return df


COLUMNS = ["created_at", "ticker", "type", "strike", "expiry", "dte", "underlying_price", "otm_pct", "total_premium",
           "ask_share", "lean", "total_size", "volume_oi_ratio", "all_opening_trades", "has_sweep", "has_floor",
           "has_multileg", "bet_like", "alert_rule"]

flow = label(fetch_flow(DAY, MIN_PREMIUM, TICKER))
print(f"{len(flow)} alerts on {DAY} at or above ${MIN_PREMIUM:,}")

## The biggest trades of the day

In [ ]:
biggest = flow.sort_values("total_premium", ascending=False)
biggest[[c for c in COLUMNS if c in biggest]].head(TOP)

### Only the ones that look like directional bets
Opening, single-leg, out of the money, 60 days or less, and leaning one way.

In [ ]:
bets = biggest[biggest["bet_like"] & (biggest["lean"] != "neutral")]
bets[[c for c in COLUMNS if c in bets]].head(TOP)

## Net directional premium by ticker
Bullish premium minus bearish premium across the day's alerts.

In [ ]:
by_ticker = (flow.groupby("ticker")
             .agg(net=("signed_premium", "sum"), gross=("total_premium", "sum"), alerts=("ticker", "size"),
                  bet_net=("signed_premium", lambda s: s[flow.loc[s.index, "bet_like"]].sum()))
             .sort_values("net"))
print("Most bullish"); display(by_ticker.tail(15)[::-1])
print("Most bearish"); display(by_ticker.head(15))

## What the stock did next
Moves from the alert day's close, signed by the trade's lean: positive means the trade's direction was right.

In [ ]:
def daily_closes(ticker: str, start: str, end: str) -> pd.Series:
    body = tradier._request("markets/history", {"symbol": ticker, "interval": "daily", "start": start, "end": end})
    days = tradier._as_list((body.get("history") or {}).get("day"))
    return pd.Series({d["date"]: float(d["close"]) for d in days}).sort_index()


def follow_up(trades: pd.DataFrame, horizons=(1, 5, 10)) -> pd.DataFrame:
    out = []
    for _, t in trades.iterrows():
        day = str(t["created_at"])[:10]
        end = (datetime.strptime(day, "%Y-%m-%d") + timedelta(days=max(horizons) * 2 + 7)).strftime("%Y-%m-%d")
        try:
            closes = daily_closes(t["ticker"], day, end)
        except Exception as exc:
            print(f"{t['ticker']}: no bars ({exc})")
            continue
        if day not in closes.index:
            continue
        sign = {"bullish": 1, "bearish": -1}.get(t["lean"], 0)
        i = list(closes.index).index(day)
        row = {"ticker": t["ticker"], "day": day, "lean": t["lean"], "premium": t["total_premium"]}
        for h in horizons:
            row[f"{h}d"] = (round(sign * (closes.iloc[i + h] / closes.iloc[i] - 1) * 100, 2)
                            if i + h < len(closes) else None)
        out.append(row)
    return pd.DataFrame(out)


follow_up(bets.head(TOP))

## The single biggest bet on each day of a range
Picks the largest bet-like, directional alert per day and scores it. Each day is one call to
Unusual Whales per page; how far back flow alerts go depends on your plan.

In [ ]:
START, END = "2026-09-01", "2026-10-16"

picks = []
for day in pd.bdate_range(START, END).strftime("%Y-%m-%d"):
    try:
        day_flow = label(fetch_flow(day, MIN_PREMIUM))
    except Exception as exc:
        print(f"{day}: {exc}")
        continue
    day_bets = day_flow[day_flow["bet_like"] & (day_flow["lean"] != "neutral")] if not day_flow.empty else day_flow
    if not day_bets.empty:
        picks.append(day_bets.sort_values("total_premium").iloc[-1])
picks = pd.DataFrame(picks)
scored = follow_up(picks)
display(scored)
print(scored[[c for c in scored if c.endswith("d")]].describe().loc[["count", "mean", "50%"]])
print("Share right:", {c: float(round((scored[c] > 0).mean(), 2)) for c in scored if c.endswith("d")})